# Multi-Model Interpretability: SHAP Analysis for Material Discovery

## 1. Abstract
To build trust in machine learning models for scientific discovery, we must explain the drivers of each prediction target. This notebook performs a comparative **SHAP (SHapley Additive exPlanations)** analysis for the three core models in our pipeline: Band Gap ($E_g$), Initial Efficiency ($PCE$), and long-term stability ($T_{80}$).

## 2. Methodology: From Black-Box to Physical Insights
The core of interpretability is understanding how the model partitions the high-dimensional physical space. We use TreeSHAP to decompose the outputs of our CatBoost ensembles.

### 2.1. Mathematical Framework: SHAP Decomposition
A prediction $f(x)$ for a single material is decomposed into a sum of contributions from each feature:

$$
f(x) = E[f(x)] + \sum_{i=1}^{M} \phi_i
$$

Where:
*   $E[f(x)]$ is the **Base Value** (the mean prediction over the background training data).
*   $\phi_i$ is the **SHAP value** for feature $i$, representing how much that feature shifted the prediction from the average.

### 2.2. Physics Interaction Logic: Non-Linearity in Trees
Machine Learning captures non-linear physical relations through depth in its decision trees. An interaction between two physical features (e.g., $t$ and $RH$) is represented by nested splits:

$$
\phi_{i,j} = \text{Interaction between } x_i \text{ and } x_j
$$

For instance, SHAP identifies that the negative impact of **Stress Humidity ($RH$)** on stability is magnified when the **Tolerance Factor ($t$)** is low, revealing the physical synergy between lattice strain and moisture-induced degradation.

In [3]:
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

# Load models
bg_model = joblib.load('../ml_models/band_gap.joblib')
pce_model = joblib.load('../ml_models/initial_pce.joblib')
t80_model = joblib.load('../ml_models/raw_t80.joblib')

# Categorical columns handling for CatBoost consistency
cat_cols = ['A_1', 'A_2', 'A_3', 'B_1', 'C_1', 'C_2', 'dimension', 
            'cell_architecture', 'etl_stack_sequence', 'htl_stack_sequence', 'backcontact_stack_sequence']

print("Models loaded for comparative SHAP analysis.")

Models loaded for comparative SHAP analysis.


## 3. Global Feature Impact Across Targets
By comparing SHAP summaries, we gain a panoramic view of the physical drivers behind each metric. The Importance Score represents the mean absolute SHAP value $|\phi_i|$, indicating the global influence of a feature on the model's output distribution.

In [ ]:
def get_impact_df(model, label):
    importances = model.get_feature_importance()
    features = model.feature_names_
    return pd.DataFrame({'Feature': features, 'Impact': importances, 'Target': label})

impacts = pd.concat([
    get_impact_df(bg_model, 'Band Gap (Eg)'),
    get_impact_df(pce_model, 'Initial Efficiency (PCE)'),
    get_impact_df(t80_model, 'Stability (T80)')
])

top_features = impacts.groupby('Feature')['Impact'].sum().sort_values(ascending=False).head(15).index
filtered_impacts = impacts[impacts['Feature'].isin(top_features)]

fig = px.bar(
    filtered_impacts, 
    x='Impact', 
    y='Feature', 
    color='Target', 
    barmode='group',
    orientation='h',
    title="Comparative Global Feature Impact: Physics Drivers across Models",
    labels={"Impact": "Importance Score (Mean |SHAP value|)"}
)
fig.update_layout(width=1000, height=800, yaxis={'categoryorder':'total ascending'}, 
                  xaxis_title="Impact on Prediction Range", 
                  legend_title="Target Property")
fig.show()

## 4. Key Scientific Inferences: Trusting the Model
1.  **Electronic Drivers:** The **Band Gap** is dominated by Anion Electronegativity (`en_C`), confirming the model learned the fundamental halide p-orbital dominance at the valence band maximum.
2.  **Architecture Drivers:** **PCE** is highly sensitive to the `ETL_stack_sequence`, indicating the model correctly identifies charge extraction at the interface as the primary efficiency hurdle.
3.  **Structural Drivers:** **Stability ($T_{80}$)** shows a massive reliance on the `tolerance_factor`, validating the model's 'understanding' of lattice strain as a driver of material failure.

## 5. Limitations of Interpretability
1.  **Correlation vs. Causality:** SHAP identifies drivers, not causal mechanisms. A high impact of `A_site_cation` might be due to its correlation with an unmeasured property like surface morphology.
2.  **Multicollinearity:** If physical descriptors are highly correlated (e.g., $r_A$ and $t$), SHAP may split the 'credit' between them, under-representing the individual importance of each.
3.  **Local vs. Global:** This summary is global. A feature might have a massive impact on a small subset of 2D materials while having zero impact on the global 3D majority.